# Generic Numba-Compatible Heap Implementation

## Parallel List Design

In [1]:
from numba import float64, types
from numba.experimental import jitclass
from numba.typed import List
import numpy as np

def make_heap_class(NodeType):
    """Create a max-heap jitclass specialized for the given NodeType.
    
    To circumvent the numba constraints of only allowing homogeneous typed tuples, the heap maintains parallel lists of float keys and associated nodes.

    The heap class is force-compiled before return for transparent performance tests.
    """

    heap_spec = [
        ('keys', types.ListType(float64)),
        ('nodes', types.ListType(NodeType))
    ]
    
    @jitclass(heap_spec)
    class Heap:
        def __init__(self):
            self.keys = List.empty_list(float64)
            self.nodes = List.empty_list(NodeType)

        def __bool__(self):
            return len(self.keys)>0

        def push(self, key, node):
            self.keys.append(key)
            self.nodes.append(node)
            self._sift_up_last()

        def pop(self):
            if len(self.keys) == 0:
                raise IndexError('pop from empty heap')
            top_key = self.keys[0]
            top_node = self.nodes[0]
            last_key = self.keys.pop()
            last_node = self.nodes.pop()
            
            if len(self.keys) == 0:
                return top_key, top_node
            
            self.keys[0] = last_key
            self.nodes[0] = last_node

            self._sift_down_root()

            return top_key, top_node
        
        def _sift_up_last(self):
            i = len(self.keys) - 1

            while i > 0:
                parent = (i - 1) // 2
                if self.keys[i] <= self.keys[parent]:
                    break
                self.keys[i], self.keys[parent] = self.keys[parent], self.keys[i]
                self.nodes[i], self.nodes[parent] = self.nodes[parent], self.nodes[i]
                i = parent

        def _sift_down_root(self):
            i = 0
            while True:
                left = 2 * i + 1
                right = 2 * i + 2
                largest = i
                if left < len(self.keys) and self.keys[left] > self.keys[largest]:
                    largest = left
                if right < len(self.keys) and self.keys[right] > self.keys[largest]:
                    largest = right
                if largest == i:
                    break
                self.keys[i], self.keys[largest] = self.keys[largest], self.keys[i]
                self.nodes[i], self.nodes[largest] = self.nodes[largest], self.nodes[i]
                i = largest
    
    _ = Heap()
    return Heap

In [2]:
from numba.types import string

def test_string_heap():
    StringHeap = make_heap_class(string)
    string_heap = StringHeap()
    assert not string_heap
    string_heap.push(1, 'one')
    assert string_heap
    string_heap.push(10, 'ten')
    string_heap.push(2, 'two')
    string_heap.push(-1, 'minus one')
    assert string_heap.pop() == (10, 'ten')
    assert string_heap.pop() == (2, 'two')
    string_heap.push(2, 'two')
    assert string_heap.pop() == (2, 'two')
    assert string_heap.pop() == (1, 'one')
    assert string_heap.pop() == (-1, 'minus one')
    assert not string_heap
    

test_string_heap()

In [3]:
from numba import njit

@njit
def push_and_pop_all(queue, sample):
    for x in sample:
        queue.push(x, x)
    while queue:
        queue.pop()

FloatHeap = make_heap_class(float64)
sample = np.random.default_rng(seed=0).normal(size=1000000)
float_heap = FloatHeap()
push_and_pop_all(float_heap, sample)

In [4]:
%timeit push_and_pop_all(float_heap, sample)

1.85 s ± 27.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Tuple Design with Generic Keys

In [5]:
def make_tuple_heap_class(KeyType, NodeType):
    """Create a max-heap jitclass specialized for the given KeyType and NodeType.
    
    Heaps store data in a list of (key, node) tuples. The heap class is force-compiled before return
    for transparent performance tests.
    """

    PairType = types.Tuple((KeyType, NodeType))
    heap_spec = [
        ('data', types.ListType(PairType))
    ]

    @jitclass(heap_spec)
    class Heap:
        def __init__(self):
            self.data = List.empty_list(PairType)

        def __bool__(self):
            return len(self.data) > 0

        def push(self, key, node):
            self.data.append((key, node))
            i = len(self.data) - 1
            while i > 0:
                parent = (i - 1) // 2
                if self.data[i][0] <= self.data[parent][0]:
                    break
                self.data[i], self.data[parent] = self.data[parent], self.data[i]
                i = parent

        def pop(self):
            if len(self.data) == 0:
                raise IndexError('pop from empty heap')
            top = self.data[0]
            last = self.data.pop()
            if len(self.data) == 0:
                return top
            self.data[0] = last
            i = 0
            while True:
                left = 2 * i + 1
                right = 2 * i + 2
                largest = i
                if left < len(self.data) and self.data[left][0] > self.data[largest][0]:
                    largest = left
                if right < len(self.data) and self.data[right][0] > self.data[largest][0]:
                    largest = right
                if largest == i:
                    break
                self.data[i], self.data[largest] = self.data[largest], self.data[i]
                i = largest
            return top

    _ = Heap()  # force compile
    return Heap

In [6]:
from numba.types import int64

def test_int_string_tuple_heap():
    StringHeap = make_tuple_heap_class(int64, string)
    string_heap = StringHeap()
    assert not string_heap
    string_heap.push(1, 'one')
    assert string_heap
    string_heap.push(10, 'ten')
    string_heap.push(2, 'two')
    string_heap.push(-1, 'minus one')
    assert string_heap.pop() == (10, 'ten')
    assert string_heap.pop() == (2, 'two')
    string_heap.push(2, 'two')
    assert string_heap.pop() == (2, 'two')
    assert string_heap.pop() == (1, 'one')
    assert string_heap.pop() == (-1, 'minus one')
    assert not string_heap

test_int_string_tuple_heap()

In [7]:
FloatFloatTupleHeap = make_tuple_heap_class(float64, float64)
# sample = np.random.default_rng(seed=0).normal(size=1000000)
float_float_tuple_heap = FloatFloatTupleHeap()
push_and_pop_all(float_float_tuple_heap, sample)

In [9]:
%timeit push_and_pop_all(float_float_tuple_heap, sample)

1.49 s ± 12.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Conclusion

Based on slightly better performance at least for large test cases and cleaner code, the tuple design is currently favored. The overhead for creating a tuple on push is likely outweighed by avoiding the parallel sifting for all but very small test cases. More systematic tests could be performed in the future, but this does not seem to be critical. The generic constructor will make future switches easy in any event.